In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# ============================================================
# 1. Ground truth SDF (unchanged)
# ============================================================
def hash2_torch(p):
    x = torch.sin(p[..., 0] * 12.9898 + p[..., 1] * 78.233) * 43758.5453
    return x - torch.floor(x)

def value_noise_2d_torch(x, z):
    xi, zi = torch.floor(x), torch.floor(z)
    xf, zf = x - xi, z - zi
    u = xf * xf * (3 - 2 * xf)
    v = zf * zf * (3 - 2 * zf)
    a = hash2_torch(torch.stack([xi,     zi    ], -1))
    b = hash2_torch(torch.stack([xi + 1, zi    ], -1))
    c = hash2_torch(torch.stack([xi,     zi + 1], -1))
    d = hash2_torch(torch.stack([xi + 1, zi + 1], -1))
    return a*(1-u)*(1-v) + b*u*(1-v) + c*(1-u)*v + d*u*v

def fbm_torch(x, z, octaves=5):
    f, amp, total, norm = 1.0, 1.0, 0.0, 0.0
    for _ in range(octaves):
        total = total + amp * value_noise_2d_torch(x * f, z * f)
        norm += amp
        f *= 2.0
        amp *= 0.5
    return total / norm

def ridged_torch(x, z, octaves=5):
    f, amp, total, norm = 1.0, 1.0, 0.0, 0.0
    for _ in range(octaves):
        n = value_noise_2d_torch(x * f, z * f)
        total = total + amp * (1.0 - torch.abs(2.0*n - 1.0))
        norm += amp
        f *= 2.0
        amp *= 0.5
    return total / norm

def gt_sdf_torch(p):
    x, y, z = p[..., 0], p[..., 1], p[..., 2]
    h = 0.3 * fbm_torch(x * 0.8, z * 0.8, octaves=4) - 0.2
    h = h + 0.05 * ridged_torch(x * 3.0, z * 3.0, octaves=3)
    sdf_ground = y - h
    cx = 0.6 * torch.sin(z * 0.5) + 0.3 * torch.sin(z * 1.3)
    dx = x - cx
    half_w = 0.4 + 0.15 * torch.sin(z * 0.7)
    horizontal = torch.abs(dx) - half_w
    canyon_depth = 0.8
    vertical = y - (h - canyon_depth)
    inside = torch.maximum(horizontal, -vertical)
    return torch.maximum(sdf_ground, -inside)



Using device: cuda
